# Notebook 2 — Evidence for each row

One row per production file at every labelled release, exactly the rows of Notebook 1's
`labels_<repo>.csv`. For each row this notebook records **which** commits, issues and PRs belong to it.
It writes no texts and no features: Notebook 3 turns these lists into the code, history and discourse
texts.

| writes (to `outputs/notebook_2_cbert/`) | one line per | used by Notebook 3 for |
|---|---|---|
| `rows_<repo>` | (file, release) | the rows and their label |
| `releases_<repo>.csv` | release | the cutoff time t and the parent release (for the split) |
| `history_commits_<repo>` | (row, commit) | which commits touched the file before t, and **the file's name in each commit** (`path_in_commit`), so its diff can be found after a rename |
| `commit_messages_<repo>` | commit | the message, time and bot flag of every commit above |
| `text_items_<repo>` | issue or PR | its creation time, bot flag and title + body |
| `text_comments_<repo>` | comment or review | its parent issue or PR, time, bot flag and text |
| `history_text_<repo>` | (row, issue or PR) | which issues and PRs are linked to the row's commits |
| `manifest.json` | file | row counts and SHA-256 of every file, for the Zenodo upload |

**Rules.**
- History is read only from commits in the release's own history (git ancestry). Merge commits carry
  no file changes and are skipped.
- A rename is followed back to the old name; an add ends the file's history.
- An issue or PR is linked to a row only if it was opened before the row's release.
- Bot flags are **recorded, not applied**: Notebook 3 leaves bot commits and bot text out.

| cell | what it does |
|---|---|
| 0 | the mined inputs, from Zenodo |
| 1–3 | setup, the rows, the releases |
| 4–7 | the commit history of each row, and a check against git |
| 8–10 | commit messages, issues/PRs/comments, and which issues and PRs belong to each row |
| 11 | check everything, save, write the manifest |

Run Notebook 1 first: this notebook reads its `labels_<repo>.csv` and `label_summary.csv`.

### Cell 0 — The mined inputs, from Zenodo
Notebooks 0–3 read what was mined from the five repos: the release grids, Quarry's outputs and a git
clone of each repo. They are published on Zenodo, [10.5281/zenodo.24182007](https://doi.org/10.5281/zenodo.24182007),
as one zip and one git bundle per repo.

For each repo this cell downloads only what is missing from `workspace_data/`:
- the zip, checked against the record's `SHA256SUMS.txt` and unpacked into `workspace_data/`, every file
  checked against the zip's `MANIFEST_<repo>.json`;
- the bundle, checked the same way and restored as the clone in `workspace_data/repos/<repo>/`.

A workspace that already holds the files and clones (for example one that ran Quarry itself) downloads
nothing. The same cell opens Notebooks 0, 1, 2 and 3, so any of them can be run first. The first run
downloads about 2.5 GB.

In [ ]:
import os, json, hashlib, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

# The workspace Quarry uses: QUARRY_HOME if set, otherwise workspace_data beside the codebase.
WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
MINED_RECORD = "24182007"                         # Zenodo record of the mined inputs
MINED_REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]
NEEDED = ["versions/rel_hist_{r}.json", "outputs/{r}/intervals_{r}.json", "outputs/{r}/refactorings_{r}.jsonl",
          "outputs/{r}/ledger_history_{r}.jsonl", "outputs/{r}/diffs_{r}.jsonl", "outputs/{r}/links_{r}.csv"]


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def have_mined_files(repo):
    """Every file the notebooks need, in outputs/<repo>/ or outputs/ (both layouts are read)."""
    def there(rel):
        p = WORKSPACE / rel.format(r=repo)
        return p.exists() or (WORKSPACE / "outputs" / p.name).exists()
    return all(there(x) for x in NEEDED)


def download(name, dest, sums):
    """One file of the record, checked against SHA256SUMS.txt before it is kept."""
    tmp = dest.with_name(dest.name + ".part")
    print(f"   downloading {name}")
    urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/{name}?download=1", tmp)
    if sha256_of(tmp) != sums[name]:
        tmp.unlink()
        raise SystemExit(f"{name}: SHA-256 differs from the record's SHA256SUMS.txt; download it again")
    tmp.replace(dest)


def fetch_mined(repos):
    downloads = WORKSPACE / "downloads"
    downloads.mkdir(parents=True, exist_ok=True)
    sums = None
    for repo in repos:
        clone = WORKSPACE / "repos" / repo
        need_files, need_clone = not have_mined_files(repo), not (clone / ".git").exists()
        if not (need_files or need_clone):
            print(f"{repo:<18}mined files and clone already in {WORKSPACE}")
            continue
        if sums is None:
            urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/SHA256SUMS.txt?download=1",
                                       downloads / "SHA256SUMS.txt")
            sums = {n: h for h, n in (l.split() for l in
                    (downloads / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines() if l.strip())}
        if need_files:
            z = downloads / f"{repo}_mined.zip"
            download(z.name, z, sums)
            with zipfile.ZipFile(z) as zf:
                manifest = json.loads(zf.read(f"MANIFEST_{repo}.json"))
                for f in manifest["files"]:
                    out = WORKSPACE / f["file"]
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(f["file"]) as src, open(out, "wb") as dst:
                        shutil.copyfileobj(src, dst, 1 << 24)
                    if sha256_of(out) != f["sha256"]:
                        raise SystemExit(f"{f['file']}: SHA-256 differs from MANIFEST_{repo}.json")
                (WORKSPACE / "versions" / f"MANIFEST_{repo}.json").write_bytes(zf.read(f"MANIFEST_{repo}.json"))
            z.unlink()
            print(f"{repo:<18}{len(manifest['files'])} mined files unpacked and checked")
        if need_clone:
            b = downloads / f"{repo}.bundle"
            download(b.name, b, sums)
            clone.mkdir(parents=True, exist_ok=True)
            for args in (["clone", "--mirror", "--quiet", str(b.resolve()), str(clone / ".git")],
                         ["-C", str(clone), "config", "--bool", "core.bare", "false"],
                         ["-C", str(clone), "-c", "core.longpaths=true", "reset", "--hard", "--quiet"]):
                r = subprocess.run(["git", *args], capture_output=True, text=True)
                if r.returncode != 0:
                    raise SystemExit(f"git {' '.join(args[:2])} failed for {repo}: {r.stderr.strip()[:300]}")
            b.unlink()
            print(f"{repo:<18}clone restored from {b.name}")


fetch_mined(MINED_REPOS)

### Cell 1 — Setup
Paths, the repositories (set `REPOS = ["commons-lang"]` to pilot on one) and shared helpers.

In [1]:
import os, re, sys, json, random, hashlib, platform, subprocess, collections
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)

WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
    else Path.cwd().parent / "workspace_data"
OUTPUTS, CLONES = WORKSPACE / "outputs", WORKSPACE / "repos"
OUT = OUTPUTS / "notebook_2_cbert"                 # everything this notebook writes goes here
OUT.mkdir(parents=True, exist_ok=True)

REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]
# REPOS = ["commons-lang"]                        # pilot on one repository first

BOT = re.compile(r"\[bot\]|dependabot|renovate|github-actions|noreply@github\.com$", re.I)


def find(repo, name):
    """A Quarry output, in outputs/<repo>/, outputs/ or versions/ -- whichever holds it."""
    for d in (OUTPUTS / repo, OUTPUTS, WORKSPACE / "versions"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found in {OUTPUTS / repo}, {OUTPUTS} or {WORKSPACE / 'versions'}")


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def norm(path):
    """A path in git's form: forward slashes, relative to the repository root."""
    p = str(path).replace("\\", "/")
    for marker in [f"/repos/{r}/" for r in REPOS]:
        if marker in p:
            p = p.split(marker, 1)[1]
    while p.startswith("./"):
        p = p[2:]
    return p.lstrip("/")


def git(repo, *args):
    """Run git in the clone; stop loudly if it fails rather than return an empty answer."""
    r = subprocess.run(["git", "-c", "core.quotepath=off", *args], cwd=CLONES / repo,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed in {repo}: {r.stderr.strip()[:300]}")
    return r.stdout


for repo in REPOS:
    if not (CLONES / repo / ".git").exists():
        raise SystemExit(f"no clone at {CLONES / repo}")
print(f"workspace {WORKSPACE}\nwriting to {OUT}\nrepositories: {', '.join(REPOS)}")

workspace E:\Thesis Project\Smell Ranker\workspace_data
writing to E:\Thesis Project\Smell Ranker\workspace_data\outputs\notebook_2_cbert
repositories: checkstyle, commons-lang, dubbo, junit-framework, questdb


### Cell 2 — The rows
Every row of Notebook 1's `labels_<repo>.csv`: release, path and label. The cell stops if a key
repeats, a value is missing, the label is not 0/1, or the count differs from Notebook 1's summary.

In [2]:
NB1 = OUTPUTS / "notebook_1_cbert"
expected = pd.read_csv(NB1 / "label_summary.csv").set_index("repo")["labelled rows"]

ROWS = {}
for repo in REPOS:
    L = pd.read_csv(NB1 / f"labels_{repo}.csv", usecols=["pos", "release", "path", "refactored"],
                    dtype={"path": str, "release": str})
    L = L.rename(columns={"pos": "release_pos", "refactored": "label"})
    L.insert(0, "repo", repo)
    L.insert(0, "row_id", L.repo + "|" + L.release + "|" + L.path)

    if L.row_id.duplicated().any():
        raise SystemExit(f"{repo}: {int(L.row_id.duplicated().sum())} repeated (release, path) keys")
    if L.isna().any().any():
        raise SystemExit(f"{repo}: missing values in the keys or the label")
    if not set(L.label.unique()) <= {0, 1}:
        raise SystemExit(f"{repo}: the label holds values other than 0 and 1")
    if len(L) != int(expected[repo]):
        raise SystemExit(f"{repo}: {len(L):,} rows, but Notebook 1's summary says {int(expected[repo]):,}")
    ROWS[repo] = L.reset_index(drop=True)

summary = pd.DataFrame([{"repo": r, "rows": len(L), "releases": L.release_pos.nunique(),
                         "refactored": int(L.label.sum()), "rate": round(L.label.mean(), 4)}
                        for r, L in ROWS.items()])
print(summary.to_string(index=False))
print(f"\ntotal rows {summary.rows.sum():,}   refactored {summary.refactored.sum():,}")

           repo   rows  releases  refactored   rate
     checkstyle  70105       201        3640 0.0519
   commons-lang   4124        28         793 0.1923
          dubbo 140453       106        5805 0.0413
junit-framework  28545        48        1590 0.0557
        questdb 218893       114       11365 0.0519

total rows 462,120   refactored 23,193


### Cell 3 — The releases
One line per release of the tree: position, tag, commit, the commit's exact time (from git) and its
**parent** release. The time is the **cutoff t**: nothing written at or after it is evidence for the
release's rows. The parent is what Notebook 3's split uses to know when a release's label became known.
The cell checks that every row's release name sits at the position the row claims.

In [3]:
RELEASES = {}
for repo in REPOS:
    d = json.loads(find(repo, f"intervals_{repo}.json").read_text(encoding="utf-8"))
    if d.get("kind") != "tree":
        raise SystemExit(f"{repo}: intervals_{repo}.json is the one-list version -- run "
                         f"build_release_tree.py --write-intervals first")
    R = pd.DataFrame([{"release_pos": r["pos"], "release": r["tag"], "sha": r["sha"],
                       "parent": r["parent"] or "", "children": len(r["children"])}
                      for r in d["releases"]]).sort_values("release_pos").reset_index(drop=True)
    R["parent_pos"] = R.parent.map(dict(zip(R.release, R.release_pos))).astype("Int64")  # empty: a root

    stamps = dict(line.split() for line in
                  git(repo, "show", "-s", "--format=%H %ct", *R.sha.unique()).split("\n") if line.strip())
    R["cutoff_ts"] = R.sha.map(lambda s: int(stamps[s]))
    R["cutoff"] = pd.to_datetime(R.cutoff_ts, unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M")
    R.insert(0, "repo", repo)

    bad = ROWS[repo][["release_pos", "release"]].drop_duplicates().merge(
        R[["release_pos", "release"]], on="release_pos", how="left", suffixes=("", "_table"))
    bad = bad[bad.release != bad.release_table]
    if len(bad):
        raise SystemExit(f"{repo}: rows name a different release than the release tree at "
                         f"{len(bad)} position(s), e.g. {bad.head(3).to_dict('records')}")
    RELEASES[repo] = R
    print(f"{repo:<18}{len(R):>4} releases   roots {int((R.parent == '').sum())}   "
          f"forks {int((R.children > 1).sum()):>2}   {R.cutoff.min()} .. {R.cutoff.max()}")

checkstyle         203 releases   roots 1   forks  1   2001-06-22 13:24 .. 2026-07-12 02:51
commons-lang        35 releases   roots 2   forks  4   2002-10-04 03:23 .. 2025-11-12 19:55
dubbo              123 releases   roots 1   forks 15   2012-06-26 06:51 .. 2026-05-17 05:43
junit-framework     61 releases   roots 1   forks 10   2017-09-10 18:07 .. 2026-07-12 12:25
questdb            119 releases   roots 1   forks  4   2014-05-06 02:09 .. 2026-06-15 16:42


### Cell 4 — What each commit changed
From the ledger: for every commit, its author's bot flag and the Java files it changed (modify, add,
rename and from where, delete).

From git itself: every rename, found with git's limit on renames per commit switched off (`-M -l0`).
A commit that moves thousands of files at once (questdb's `nfsdb-core` to `core`, checkstyle's
`src/checkstyle` to `src/main/java`) exceeds git's default limit, and the ledger then records those moves
as a delete plus an add, which would cut a file's history at the move.

In [4]:
def rename_paths(p):
    """git's numstat rename notation -> (old, new): 'a/{x => y}/F.java' or 'old/F.java => new/F.java'."""
    if "{" in p and " => " in p:
        pre, rest = p.split("{", 1)
        mid, post = rest.split("}", 1)
        a, b = mid.split(" => ", 1)
        return norm((pre + a + post).replace("//", "/")), norm((pre + b + post).replace("//", "/"))
    a, b = p.split(" => ", 1)
    return norm(a), norm(b)


COMMITS, GIT_RENAMES = {}, {}
for repo in REPOS:
    led = {}
    for rec in read_jsonl(find(repo, f"ledger_history_{repo}.jsonl")):
        email = (rec.get("author_email") or "").strip().lower()
        mods = [(norm(m["file"]) if m.get("file") else None,
                 norm(m["old_file"]) if m.get("old_file") else None,
                 m.get("change_type")) for m in rec.get("modifications") or []]
        led[rec["sha"]] = {"bot": bool(BOT.search(email)), "mods": mods}
    COMMITS[repo] = led

    renames = {}                                   # (commit, new path) -> old path
    sha = None
    for line in git(repo, "log", "--no-merges", "-M", "-l0", "--numstat", "--format=@%H",
                    *RELEASES[repo].sha.unique().tolist(), "--", "*.java").splitlines():
        if line.startswith("@"):
            sha = line[1:]
        elif " => " in line:
            old, new = rename_paths(line.split("\t", 2)[2])
            renames[(sha, new)] = old
    GIT_RENAMES[repo] = renames

    print(f"{repo:<18}{len(led):>7,} commits in the ledger ({sum(c['bot'] for c in led.values()):,} by bots)   "
          f"git renames {len(renames):,}")

checkstyle         17,185 commits in the ledger (1,327 by bots)   git renames 4,912
commons-lang        9,883 commits in the ledger (355 by bots)   git renames 660
dubbo               9,956 commits in the ledger (1,092 by bots)   git renames 5,813
junit-framework    12,172 commits in the ledger (1,039 by bots)   git renames 2,389
questdb             6,340 commits in the ledger (145 by bots)   git renames 6,578


### Cell 5 — Each release's history
For every release that has rows: the commits in its history, newest first (`git rev-list --topo-order`),
whatever branch the release sits on, and which of them are merges (skipped in cell 6).

In [5]:
HISTORY, SHA_OF, MERGE_IDS = {}, {}, {}
for repo in REPOS:
    ids = {}                                       # sha -> small integer, to keep memory low
    R = RELEASES[repo].set_index("release_pos")
    hist = {}
    for pos in sorted(ROWS[repo].release_pos.unique()):
        shas = git(repo, "rev-list", "--topo-order", R.sha[pos]).split()
        hist[pos] = np.array([ids.setdefault(s, len(ids)) for s in shas], dtype=np.int32)
    merges = set(git(repo, "rev-list", "--merges", *R.sha[sorted(hist)].tolist()).split())
    HISTORY[repo] = hist
    MERGE_IDS[repo] = {ids[s] for s in merges if s in ids}
    SHA_OF[repo] = np.array(sorted(ids, key=ids.get))

    missing = [s for s in ids if s not in COMMITS[repo]]
    lengths = [len(h) for h in hist.values()]
    print(f"{repo:<18}{len(hist):>4} releases   history {min(lengths):,}..{max(lengths):,} commits   "
          f"merge commits {len(MERGE_IDS[repo]):,}   history commits missing from the ledger: {len(missing)}")

checkstyle         201 releases   history 2..16,894 commits   merge commits 19   history commits missing from the ledger: 0
commons-lang        28 releases   history 152..8,872 commits   merge commits 226   history commits missing from the ledger: 0
dubbo              106 releases   history 1,443..8,464 commits   merge commits 517   history commits missing from the ledger: 4
junit-framework     48 releases   history 3,687..10,778 commits   merge commits 114   history commits missing from the ledger: 0
questdb            114 releases   history 9..6,097 commits   merge commits 60   history commits missing from the ledger: 0


### Cell 6 — Which commits touched each row's file, and under which name
For each release, walk its history from newest to oldest, starting from the files at that release.
Each time a commit touches a tracked file, one line is recorded: the row, the commit, and
**`path_in_commit`**, the name the file had in that commit (the name its diff is filed under).
When a commit **renames** the file (by the ledger, or by git when the ledger saw an add), older commits
are followed under the old name; when a commit **adds** it, the walk stops for that file (anything older
at the same path was another file).

In [6]:
TOUCH = {}
for repo in REPOS:
    led, sha_of, grn = COMMITS[repo], SHA_OF[repo], GIT_RENAMES[repo]
    recovered = 0
    rows_, cids, paths_ = [], [], []
    for pos, g in ROWS[repo].groupby("release_pos"):
        alias = dict(zip(g.path, g.index))         # name now -> row (index in ROWS[repo])
        for cid in HISTORY[repo][pos]:             # newest first
            if not alias:
                break
            if cid in MERGE_IDS[repo]:             # merges: their changes are the branch's commits
                continue
            sha = sha_of[cid]
            rec = led.get(sha)
            if rec is None:
                continue
            for new, old, kind in rec["mods"]:
                row = alias.get(new)
                if row is None or kind == "DELETE":
                    continue
                if kind != "RENAME" and (sha, new) in grn:   # a rename the ledger saw as an add
                    old, kind = grn[(sha, new)], "RENAME"
                    recovered += 1
                rows_.append(row); cids.append(cid); paths_.append(new)   # new = its name in this commit
                if kind == "RENAME" and old:
                    del alias[new]
                    alias[old] = row
                elif kind == "ADD":
                    del alias[new]
    T = pd.DataFrame({"row": np.array(rows_, dtype=np.int32), "cid": np.array(cids, dtype=np.int32),
                      "path_in_commit": paths_})
    T = T.drop_duplicates(["row", "cid"], keep="first").reset_index(drop=True)   # a file listed twice
    TOUCH[repo] = T
    renamed = int((T.path_in_commit.to_numpy() != ROWS[repo].path.to_numpy()[T.row]).sum())
    print(f"{repo:<18}{len(T):>10,} (row, commit) lines   rows touched {T.row.nunique():,} of {len(ROWS[repo]):,}   "
          f"lines under an older name {renamed:,}   renames recovered from git {recovered:,}")

checkstyle         3,384,143 (row, commit) lines   rows touched 70,105 of 70,105   lines under an older name 1,285,788   renames recovered from git 0
commons-lang         142,470 (row, commit) lines   rows touched 4,124 of 4,124   lines under an older name 54,941   renames recovered from git 0
dubbo              1,452,067 (row, commit) lines   rows touched 140,453 of 140,453   lines under an older name 575,735   renames recovered from git 12
junit-framework      632,801 (row, commit) lines   rows touched 28,545 of 28,545   lines under an older name 196,529   renames recovered from git 0
questdb            2,921,802 (row, commit) lines   rows touched 218,893 of 218,893   lines under an older name 872,878   renames recovered from git 0


### Cell 7 — Check the commit lists against git
For 200 random rows per repository, compare the commits found in cell 6 with git's own list
(`git log --follow --no-merges <release> -- <path>`), **cut where git says the file began**: the commit
that added it (`A`) or created it as a copy of another file (`C`). Without the cut, `git log --follow`
carries on past a copy into the *source* file's history, commits that never touched this file.
The table reports how many sampled rows have identical commit lists, the mean overlap, and the
commits only one side found.

In [7]:
rng = random.Random(0)
CHECK = []
for repo in REPOS:
    T, L, sha_of = TOUCH[repo], ROWS[repo], SHA_OF[repo]
    by_row = T.groupby("row").cid.apply(set)
    rel_sha = dict(zip(RELEASES[repo].release_pos, RELEASES[repo].sha))
    same, overlap, ours_only, git_only = 0, [], 0, 0
    for row in rng.sample(range(len(L)), min(200, len(L))):
        ours = {sha_of[c] for c in by_row.get(row, set())}
        entries, sha = [], None                    # (commit, git status), newest first
        for line in git(repo, "log", "--follow", "--no-merges", "--name-status", "--format=@%H",
                        rel_sha[L.release_pos[row]], "--", L.path[row]).splitlines():
            if line.startswith("@"):
                sha = line[1:]
            elif line and sha and line[0] in "AMDRCT":
                entries.append((sha, line[0]))
                sha = None
        theirs = set()
        for s, st in entries:                      # stop where the file began
            theirs.add(s)
            if st in ("A", "C"):
                break
        same += ours == theirs
        overlap.append(len(ours & theirs) / max(1, len(ours | theirs)))
        ours_only += len(ours - theirs)
        git_only += len(theirs - ours)
    n = len(overlap)
    CHECK.append({"repo": repo, "sampled rows": n, "identical": same,
                  "identical %": round(100 * same / n, 1), "mean overlap": round(float(np.mean(overlap)), 3),
                  "commits only we found": ours_only, "commits only git found": git_only})
print(pd.DataFrame(CHECK).to_string(index=False))

           repo  sampled rows  identical  identical %  mean overlap  commits only we found  commits only git found
     checkstyle           200        200        100.0         1.000                      0                       0
   commons-lang           200        200        100.0         1.000                      0                       0
          dubbo           200        182         91.0         0.971                     19                      19
junit-framework           200        196         98.0         0.997                      1                      12
        questdb           200        187         93.5         0.986                     15                      19


### Cell 8 — Commit messages
The message, time and bot flag of every commit in cell 6's lists, read from git itself (one call per
repository), so no commit lacks its message.

In [8]:
MESSAGES = {}
for repo in REPOS:
    led = COMMITS[repo]
    needed = set(SHA_OF[repo][np.unique(TOUCH[repo].cid.to_numpy())])
    out = git(repo, "log", "--no-merges", "--format=%x1e%H%x1f%ct%x1f%B",
              *RELEASES[repo].sha.unique().tolist())
    rows = []
    for rec in out.split("\x1e"):
        if not rec.strip():
            continue
        sha, ts, msg = rec.split("\x1f", 2)
        if sha in needed:
            rows.append((sha, int(ts), bool(led.get(sha, {}).get("bot", False)), msg.strip()))
    M = pd.DataFrame(rows, columns=["sha", "ts", "bot", "message"])
    MESSAGES[repo] = M
    print(f"{repo:<18}commits in row histories {len(needed):,}   with a message {len(M):,}   "
          f"by bots {int(M.bot.sum()):,}   empty messages {int((M.message == '').sum()):,}")

checkstyle        commits in row histories 5,809   with a message 5,809   by bots 17   empty messages 1
commons-lang      commits in row histories 4,482   with a message 4,482   by bots 0   empty messages 0
dubbo             commits in row histories 5,592   with a message 5,592   by bots 2   empty messages 0
junit-framework   commits in row histories 4,200   with a message 4,200   by bots 3   empty messages 0
questdb           commits in row histories 4,213   with a message 4,213   by bots 1   empty messages 0


### Cell 9 — Issues, PRs, comments and reviews
One line per issue and per PR (kind, number, created time, bot, title + body) and one line per comment
or review (its issue or PR, its own id, time, bot, text). GitHub serves a PR's conversation through the
issue-comments endpoint, so every comment is attached to its **real** parent: a PR when its number is a
PR, an issue when it is an issue. Comments whose parent was not mined are dropped and counted. Reviews
always belong to a PR. Empty texts (a bare approval) are dropped. Quarry mined closed issues and PRs only.

In [9]:
def channel_records(repo):
    seen, recs = set(), collections.defaultdict(list)
    for d in (OUTPUTS / repo, OUTPUTS):
        for p in sorted(d.glob(f"channel_*_{repo}.jsonl")):
            if p.name not in seen:
                seen.add(p.name)
                for r in read_jsonl(p):
                    recs[r.get("channel")].append(r)
    return recs


def author(r):
    return (r.get("author") or "").strip()


ITEMS, COMMENTS, COMMENT_PARENT = {}, {}, {}
for repo in REPOS:
    ch = channel_records(repo)
    items = [("issue", r["record_id"], r.get("timestamp"), bool(BOT.search(author(r))),
              (r.get("text") or "").strip()) for r in ch.get("github.issue", [])] + \
            [("pr", r["record_id"], r.get("timestamp"), bool(BOT.search(author(r))),
              (r.get("text") or "").strip()) for r in ch.get("github.pr", [])]
    I = pd.DataFrame(items, columns=["kind", "number", "ts", "bot", "text"])
    issues, prs = set(I.number[I.kind == "issue"]), set(I.number[I.kind == "pr"])

    parent, rows, lost = {}, [], 0
    for r in ch.get("github.comment", []):
        p = r.get("parent_id")
        kind = "pr" if p in prs else "issue" if p in issues else None
        if kind is None:
            lost += 1
            continue
        parent[("github.comment", r["record_id"])] = (kind, p)
        rows.append((kind, p, "comment", r["record_id"], r.get("timestamp"), bool(BOT.search(author(r))),
                     (r.get("text") or "").strip()))
    for r in ch.get("github.pr_review", []):
        p = r.get("parent_id")
        if p not in prs:
            lost += 1
            continue
        parent[("github.pr_review", r["record_id"])] = ("pr", p)
        rows.append(("pr", p, "review", r["record_id"], r.get("timestamp"), bool(BOT.search(author(r))),
                     (r.get("text") or "").strip()))
    C = pd.DataFrame(rows, columns=["kind", "number", "source", "record_id", "ts", "bot", "text"])
    C = C[C.text != ""].reset_index(drop=True)

    ITEMS[repo], COMMENTS[repo], COMMENT_PARENT[repo] = I, C, parent
    print(f"{repo:<18}issues {len(issues):,}   PRs {len(prs):,}   comments+reviews kept {len(C):,}   "
          f"parent not mined {lost:,}   by bots: items {int(I.bot.sum()):,}, comments+reviews {int(C.bot.sum()):,}")

checkstyle        issues 5,586   PRs 14,512   comments+reviews kept 129,274   parent not mined 5,353   by bots: items 1,172, comments+reviews 5,460
commons-lang      issues 0   PRs 1,703   comments+reviews kept 5,605   parent not mined 574   by bots: items 420, comments+reviews 67
dubbo             issues 6,588   PRs 8,409   comments+reviews kept 41,186   parent not mined 3,520   by bots: items 1,227, comments+reviews 2,033
junit-framework   issues 2,551   PRs 2,994   comments+reviews kept 30,199   parent not mined 1,149   by bots: items 1,223, comments+reviews 1,937
questdb           issues 1,636   PRs 4,740   comments+reviews kept 29,732   parent not mined 2,608   by bots: items 176, comments+reviews 3,020


### Cell 10 — Which issues and PRs belong to each row
An issue or PR belongs to a row when it is linked (`links_<repo>.csv`) to one of the row's commits:
directly, or through one of its comments or reviews. **It is kept only if it was opened before the
row's release**: an issue opened later is future text, even when it mentions an old commit. Its
comments and reviews are cut at the release time in Notebook 3, one by one.

In [10]:
TEXT_LINKS = {}
for repo in REPOS:
    I, parent = ITEMS[repo], COMMENT_PARENT[repo]
    created = {(k, n): t for k, n, t in zip(I.kind, I.number, I.ts)}
    links = pd.read_csv(find(repo, f"links_{repo}.csv"), dtype=str)

    by_sha = collections.defaultdict(set)          # commit -> {(kind, number)}
    for chan, sid, sha in zip(links.source_channel, links.source_id, links.target_sha):
        try:
            sid = int(sid)
        except ValueError:
            continue
        item = ("issue", sid) if chan == "github.issue" else ("pr", sid) if chan == "github.pr" \
            else parent.get((chan, sid))
        if item in created:
            by_sha[sha].add(item)

    T, L, sha_of = TOUCH[repo], ROWS[repo], SHA_OF[repo]
    cutoff = L.release_pos.map(RELEASES[repo].set_index("release_pos").cutoff_ts).to_numpy()
    pairs, future = set(), 0
    for row, cid in zip(T.row.to_numpy(), T.cid.to_numpy()):
        for item in by_sha.get(sha_of[cid], ()):
            t = created[item]
            if t is None or pd.isna(t) or t >= cutoff[row]:
                future += 1
                continue
            pairs.add((row, *item))
    H = pd.DataFrame([(L.row_id.iat[r], k, n) for r, k, n in sorted(pairs)],
                     columns=["row_id", "kind", "number"])
    TEXT_LINKS[repo] = H
    n = len(L)
    print(f"{repo:<18}rows with an issue {H[H.kind == 'issue'].row_id.nunique() / n:6.1%}   "
          f"with a PR {H[H.kind == 'pr'].row_id.nunique() / n:6.1%}   "
          f"links dropped as opened after the release {future:,}")

checkstyle        rows with an issue  92.3%   with a PR  80.4%   links dropped as opened after the release 19
commons-lang      rows with an issue   0.0%   with a PR  45.0%   links dropped as opened after the release 7
dubbo             rows with an issue  51.7%   with a PR  87.1%   links dropped as opened after the release 0
junit-framework   rows with an issue  95.3%   with a PR  82.8%   links dropped as opened after the release 0
questdb           rows with an issue  62.7%   with a PR  93.4%   links dropped as opened after the release 0


### Cell 11 — Check, save, and write the manifest
Stops if any of these fails, before anything is written:
- every row has at least one commit line, and one line under its own name (the newest touch is always
  under the name the file has at the release);
- no (row, commit) pair repeats, and every `path_in_commit` is a `.java` path;
- every commit has a message line, every linked issue or PR is in `text_items`, and every comment's
  parent is in `text_items`;
- every linked issue or PR was opened before its row's cutoff.

Then writes every table and `manifest.json`: each file's name, line count and SHA-256, the versions
used and each clone's commit, so the Zenodo upload can be checked byte for byte.

In [11]:
def save(df, name):
    """Parquet when pyarrow is installed (smaller, keeps types); compressed CSV otherwise."""
    try:
        df.to_parquet(OUT / f"{name}.parquet", index=False)
        return OUT / f"{name}.parquet"
    except ImportError:
        df.to_csv(OUT / f"{name}.csv.gz", index=False)
        return OUT / f"{name}.csv.gz"


TABLES = {}
for repo in REPOS:
    L, T, M, I, C, H = ROWS[repo], TOUCH[repo], MESSAGES[repo], ITEMS[repo], COMMENTS[repo], TEXT_LINKS[repo]
    sha_of = SHA_OF[repo]
    HC = pd.DataFrame({"row_id": L.row_id.to_numpy()[T.row], "sha": sha_of[T.cid],
                       "path_in_commit": T.path_in_commit})

    untouched = len(L) - T.row.nunique()
    own = set(T.row[T.path_in_commit.to_numpy() == L.path.to_numpy()[T.row]])
    problems = {
        "rows without any commit": untouched,
        "rows without a line under their own name": len(set(T.row) - own),
        "repeated (row, commit)": int(HC.duplicated(["row_id", "sha"]).sum()),
        "path_in_commit not .java": int((~HC.path_in_commit.str.endswith(".java")).sum()),
        "commits without a message line": len(set(HC.sha) - set(M.sha)),
        "linked items not in text_items": len(set(zip(H.kind, H.number)) - set(zip(I.kind, I.number))),
        "comments whose parent is not in text_items": len(set(zip(C.kind, C.number)) - set(zip(I.kind, I.number))),
    }
    created = dict(zip(zip(I.kind, I.number), I.ts))
    cut = dict(zip(L.row_id, L.release_pos.map(RELEASES[repo].set_index("release_pos").cutoff_ts)))
    problems["links opened at or after the cutoff"] = int(sum(
        created[(k, n)] >= cut[r] for r, k, n in zip(H.row_id, H.kind, H.number)))
    bad = {k: v for k, v in problems.items() if v}
    if bad:
        raise SystemExit(f"{repo}: {bad}")

    TABLES[repo] = {"rows": L[["row_id", "repo", "release", "release_pos", "path", "label"]],
                    "releases": RELEASES[repo], "history_commits": HC, "commit_messages": M,
                    "text_items": I, "text_comments": C, "history_text": H}
    print(f"{repo:<18}all checks passed")

manifest = {"notebook": "notebook_2_CBERT", "files": []}
for repo, tables in TABLES.items():
    for name, df in tables.items():
        path = save(df, f"{name}_{repo}") if name != "releases" else OUT / f"releases_{repo}.csv"
        if name == "releases":
            df.to_csv(path, index=False)
        manifest["files"].append({"file": path.name, "repo": repo, "table": name, "lines": len(df),
                                  "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})

manifest["versions"] = {"python": platform.python_version(), "pandas": pd.__version__,
                        "numpy": np.__version__, "git": git(REPOS[0], "--version").strip()}
manifest["clones"] = {r: git(r, "rev-parse", "HEAD").strip() for r in REPOS}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")

S = pd.DataFrame(manifest["files"]).pivot(index="repo", columns="table", values="lines")
print("\nLINES PER TABLE\n")
print(S[["rows", "releases", "history_commits", "commit_messages", "text_items", "text_comments",
         "history_text"]].to_string())
print(f"\nwrote {len(manifest['files'])} tables and manifest.json to {OUT}")

checkstyle        all checks passed
commons-lang      all checks passed
dubbo             all checks passed
junit-framework   all checks passed
questdb           all checks passed

LINES PER TABLE

table              rows  releases  history_commits  commit_messages  text_items  text_comments  history_text
repo                                                                                                        
checkstyle        70105       203          3384143             5809       20098         129274       2090736
commons-lang       4124        35           142470             4482        1703           5605          6806
dubbo            140453       123          1452067             5592       14997          41186       1241483
junit-framework   28545        61           632801             4200        5545          30199        251274
questdb          218893       119          2921802             4213        6376          29732       2322773

wrote 35 tables and manifest.json to E